# EDA on ultimate marathon data to explore deeper what should be cleaned

In [0]:
df_um = spark.sql("SELECT * FROM marathos.bronze.raw_ultra_marathon")

display(df_um)

Databricks visualization. Run in Databricks to view.

In [0]:
df_um = spark.sql("SELECT * FROM marathos.bronze.raw_ultra_marathon")

display(df_um)

In [0]:
# saw club name with unsupported letters, investigate more

df_club_athletes = df_um.groupBy("Athlete club") \
                        .count() \
                        .withColumnRenamed("count", "amount") \
                        .orderBy("count", ascending = True)
                                            
df_club_athletes.limit(5).display()

In [0]:
%sql
-- see if an athlete have been in different clubs

SELECT DISTINCT
    `Athlete ID`,
    `Athlete club`
FROM 
    marathos.bronze.raw_ultra_marathon
ORDER BY `Athlete ID`
LIMIT 5;


## Check similarities and differences

In [0]:
%sql

-- See if any gender and age category mismatch

SELECT
    `Athlete gender`,
    `Athlete age category`,
    COUNT(*) as amount_rows
FROM 
    marathos.bronze.raw_ultra_marathon
WHERE 
    (`Athlete gender` = 'M' AND LEFT(`Athlete age category`, 1) != 'M')
    OR
    (`Athlete gender` = 'F' AND LEFT(`Athlete age category`, 1) != 'W')
GROUP BY
    `Athlete gender`,
    `Athlete age category`;

In [0]:
%sql
-- See if year of event and event dates matches
-- Collect year from first date if multiple

SELECT
    `Year of event`,
    SUBSTRING(`Event dates`, 7, 4) AS extract_start_year,
    `Event dates`,
    COUNT(*) as amount_rows
FROM 
    marathos.bronze.raw_ultra_marathon
WHERE 
    (TRY_CAST(SUBSTRING(`Event dates`, 7, 4) AS BIGINT) != `Year of event`)
GROUP BY
    `Year of event`,
    `Event dates`;

In [0]:
df_country = spark.sql("SELECT * FROM marathos.bronze.raw_country_reference")

df_country.limit(5).display()

In [0]:
%sql
-- Same code with matches to see it works

SELECT
    `Year of event`,
    SUBSTRING(`Event dates`, 7, 4) AS extract_start_year,
    `Event dates`
FROM 
    marathos.bronze.raw_ultra_marathon
WHERE 
    (TRY_CAST(SUBSTRING(`Event dates`, 7, 4) AS BIGINT) = `Year of event`)
GROUP BY
    `Year of event`,
    `Event dates`
LIMIT 5;

In [0]:
%sql
-- compare Event distance/length and Athlete performance
-- check if km or mi = h

SELECT
  `Event distance/length`,
  `Athlete performance`
FROM
  marathos.bronze.raw_ultra_marathon
WHERE
  (
    (
      RIGHT(`Event distance/length`, 2) = 'km'
      OR RIGHT(`Event distance/length`, 2) = 'mi'
    )
    AND RIGHT(`Athlete performance`, 1) != 'h'
  )
  OR (
    RIGHT(`Event distance/length`, 1) = 'h'
    AND (RIGHT(`Athlete performance`, 2) != 'km')
  )
  OR (
    RIGHT(`Athlete performance`, 2) = 'km'
    AND RIGHT(`Event distance/length`, 1) != 'h'
  )
  OR (
    RIGHT(`Athlete performance`, 1) = 'h'
    AND (
      RIGHT(`Event distance/length`, 2) != 'km'
      AND RIGHT(`Event distance/length`, 2) != 'mi'
    )
  )
GROUP BY
  `Event distance/length`,
  `Athlete performance`
LIMIT 5;

In [0]:
%sql

SELECT
    `Event dates`,
    `Event name`,
    `Event distance/length`,
    `Athlete performance`
FROM
    marathos.bronze.raw_ultra_marathon
WHERE `Event distance/length` ILIKE "%etapp%"
GROUP BY 
    `Event dates`,
    `Event name`,
    `Event distance/length`,
    `Athlete performance`
ORDER BY 
    `Event name`, `Event distance/length`
LIMIT 5;

In [0]:
%sql
-- count age from year of event and year of birth
-- check how many are under 18 and over 80

SELECT
  `Year of event`,
  `Athlete year of birth`,
  (`Year of event` - `Athlete year of birth`) AS athlete_age
FROM
  marathos.bronze.raw_ultra_marathon
WHERE
  (`year of event` - `Athlete year of birth`) < 18
  OR (`year of event` - `Athlete year of birth`) > 80
GROUP BY 
    `Year of event`,
    `Athlete year of birth`
  LIMIT 5;

### Summary of things to clean 

#### Country
- break out country from "Event name"
- join with country code tables -> get right match -> switch to country
- for country codes without match -> "unknown" or "missing"?

#### Extract/split 
- extract country from Event name
- split performace between h and km
- split distance/length between digits and kind (km or h) for possibility to calculate

#### Calculate
- km/h and replace athlete avreage speed
- mi (american miles) to km in Event distance/length

#### Nulls
- Athlete club - "none/unknown"
- Athlete country -> "unknown"
- Athlete gender -> "unknown"
- Athlete age category -> "unknown
- Athlete average speed -> remove row

#### Data types
- Event dates -> date
- Athlete average speed -> float/double

#### Rename columns
- convert to snake_case and lower case

#### Remove
- rows with d (days) as length/time
- rows where age (year of event - year of birth) is less than 10 and more then 90,
- rows where Athlete gender and Athlete age group (gender) does not match
- rows with etapp (all rows have 'event date' dd.-dd.mm.yy, the different etaps have different names, hard to split distance/length and performans over several dates)

#### Create ID
- event id
- date id
- result id

#### Other insights
- change from UTF-8 -> latin1 for more supported letters

#### Drop
- event_distance_length
- athlete_performance
- event_name
- athlete_average_speed
- event_dates

# Cleaning

## Standardize column names

In [0]:
import re

# 
def to_snake_case(name):
    name = name.strip().casefold()
    name = re.sub(r"[/]+", "_", name)
    name = re.sub(r"[\s]+", "_", name)
    return name

def rename_columns_to_snake_case(df_um):
    new_columns = [to_snake_case(column) for column in df_um.columns]
    return df_um.toDF(*new_columns)

df_column_alias = rename_columns_to_snake_case(df_um)

df_column_alias.limit(5).display()

## Remove unwanted rows

In [0]:
from pyspark.sql import functions as sf
from pyspark.sql.functions import to_timestamp, col

# clean data to only match km/mi to h 
df = df_column_alias.filter(
    (
        sf.lower(sf.col("event_distance_length")).rlike("(km|mi)$")
        & sf.lower(sf.col("athlete_performance")).rlike("h$")
        & ~sf.lower(sf.col("athlete_performance")).rlike("d")
    )
    | (
        sf.lower(sf.col("event_distance_length")).rlike("h$")
        & sf.lower(sf.col("athlete_performance")).rlike("km$")
    )
    | (
        sf.lower(sf.col("athlete_performance")).rlike("km$")
        & sf.lower(sf.col("event_distance_length")).rlike("h$")
    )
    | (
        sf.lower(sf.col("athlete_performance")).rlike("h$")
        & sf.lower(sf.col("event_distance_length")).rlike("(km|mi)$")
    )
)


df.select("event_distance_length", "athlete_performance").limit(3).display()

## Extract/Split values

In [0]:
from pyspark.sql import functions as sf
from pyspark.sql.functions import col

# extract value and type from event_distance_length to enable calcualtions and data type change
df = (
    df.withColumn(
        "event_distance_length_value",
        sf.regexp_extract(col("event_distance_length"), r"^([0-9]+)", 1),
    )
    .withColumn(
        "event_distance_length_type",
        sf.regexp_extract(col("event_distance_length"), r"([a-zA-Z]+)", 1),
    ).withColumn(
        "event_distance_length_value",
        sf.when(
            sf.col("event_distance_length_type") == "mi",
            sf.col("event_distance_length_value").cast("double") * 1.609344
        ).otherwise(
            sf.col("event_distance_length_value").cast("double")
        )
    ).withColumn(
        "event_distance_length_type",
        sf.when(
            sf.col("event_distance_length_type") == "mi",
            sf.lit("km")
        ).otherwise(
            sf.col("event_distance_length_type")
        )
    )  
)

df.select("event_distance_length", "event_distance_length_value", "event_distance_length_type").limit(5).display()

In [0]:
# extract different values from athlete_performance to enable calcualtions and data type change
df = df.withColumn(
    "athlete_performance_value",
    sf.regexp_extract(col("athlete_performance"), r"([0-9,.;:]+)\s*", 1)
).withColumn(
    "athlete_performance_type",
    sf.regexp_extract(col("athlete_performance"), r"([a-zA-Z]+)", 1)
).withColumn(
    "athlete_performance_value",
    sf.when(
        sf.col("athlete_performance_value").contains(":"),
        sf.round(((
            sf.expr("try_cast(substring_index(athlete_performance_value, ':', 1) as int)") * 3600 +
            sf.expr("try_cast(substring_index(substring_index(athlete_performance_value, ':', 2), ':', -1) as int)") * 60 +
            sf.expr("try_cast(substring_index(athlete_performance_value, ':', -1) as int)")
        ) / 3600), 5).cast("double")
    ).otherwise(
        sf.expr("try_cast(athlete_performance_value as double)")
    )  
)

df.select("athlete_performance", "athlete_performance_value", "athlete_performance_type").limit(3).display()

In [0]:
# extract country code from event_name to match country code to country name
df = df.withColumn(
    "name_of_event",
    sf.regexp_extract(col("event_name"), r"(.*[a-zA-Z]+)\s*\(", 1)
).withColumn(
    "event_country_code",
    sf.regexp_extract(col("event_name"), r"\(([a-zA-Z]+)\)", 1)
)

df.select("event_name", "name_of_event", "event_country_code").limit(3).display()

In [0]:
# split dates from event_dates to change data type and easier search
df = df.withColumn(
    "event_start_date",
    sf.when(
        # got help from LLM how to check and concatenate
        sf.col("event_dates").rlike(r"^\d{2}\.-\d{2}\."),
        sf.concat(
            sf.substring(sf.col("event_dates"), 1, 3),
            sf.substring(sf.col("event_dates"), 8, 8),
        ),
    ).otherwise(sf.substring(sf.col("event_dates"), 1, 10)),
).withColumn(
    "event_start_date",
    sf.to_date(sf.col("event_start_date"), "dd.MM.yyyy")
).withColumn(
    "event_end_date",
    sf.when(
        sf.col("event_dates").contains("-"),
        sf.regexp_extract(col("event_dates"), r"\-([0-9,.;:]+)\s*", 1)
    ).otherwise(sf.col("event_dates"))
).withColumn(
    "event_end_date",
    sf.to_date(sf.col("event_end_date"), "dd.MM.yyyy")
)

df.select("event_dates","event_start_date", "event_end_date").limit(3).display()

## Change datatypes

In [0]:
df.limit(3).display()


## Calculation

- km/h and replace athlete avreage speed
- age

## Standardize values
- Athlete club - "none/unknown"
- Athlete country -> "unknown"
- Athlete gender -> "unknown"
- Athlete age category -> "unknown

## Validate rows based on logic
- age>10 & age<95 
- avg_speed > ????
- m = m, f = w or f

## Join tables to obt (one big table)

## Create IDs
- event id
- date id
- result id

## Drop columns
- event_distance_length
- athlete_performance
- event_name
- athlete_average_speed